In [ ]:
import torch
import torch.nn as nn
from typing import List, Tuple, Dict, Any, Union, Optional
import copy

In [ ]:

class TiedHeadMLP(nn.Module):
    """
    Use same linear weights applied to all attention heads
    """
    def __init__(self, 
                 num_heads: int,
                 head_dim: int,     # input dim
                 feature_dim: int,  # output dim
                 dtype: torch.dtype,
                 device: torch.device,
                 skip_connection: bool = True,
                 bias: bool = False,
                 zero_init: bool = True):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = head_dim
        self.feature_dim = feature_dim
        self.dtype = dtype
        self.device = device
        self.skip_connection = skip_connection
        self.zero_init = zero_init
        self.init_weights_()
        
        if self.zero_init: 
            self.zero_init_with_skip_() if self.skip_connection else self.zero_init_()

        if self.skip_connection:
            assertion_fail = f'If self.skip_connection we need self.head_dim == self.feature_dim but self.head_dim is {self.head_dim} != self.feature_dim is {self.feature_dim}'
            assert self.head_dim == self.feature_dim, assertion_fail

    def zero_init_with_skip_(self):
        with torch.no_grad():
            nn.init.zeros_(self.layer.weight)

    def zero_init_(self_):
        with torch.no_grad():
            nn.init.eye_(self.layer.weight)

    def init_weights_(self):
        self.layer = nn.Linear(self.head_dim, self.feature_dim, bias=False,
                               dtype=self.dtype, device=self.device)

    def forward(self, x: torch.Tensor):
        """Assume x.shape is b h l d"""
        return x + self.layer(x) if self.skip_connection else self.layer(x)

class UntiedHeadMLP(TiedHeadMLP):
    """
    Use different weights per head
    """
    def init_weights_(self):
        self.layer = nn.Conv1d(in_channels=self.head_dim * self.num_heads,
                               out_channels=self.feature_dim * self.num_heads,
                               kernel_size=1, groups=self.num_heads,
                               bias=False, dtype=self.dtype, device=self.device)

    def zero_init_(self):
        with torch.no_grad():
            nn.init.eye_(self.layer.weight[..., 0])

    def _forward(self, x: torch.Tensor):
        b, h, l, d = x.shape
        x = rearrange(x, 'b h l d -> b (h d) l', h=self.num_heads)
        x = self.layer(x)
        x = rearrange(x, 'b (h d) l -> b h l d', h=self.num_heads)
        return x

    def forward(self, x: torch.Tensor):
        """Assume x.shape is b h l d"""
        return x + self._forward(x) if self.skip_connection else self._forward(x)

class UntiedHeadEinsumMLP(UntiedHeadMLP):
    """
    Alternate implementation with untied heads that uses einsum
    """
    def __init__(self, 
                 normal_init: bool = False, 
                 *args: any, **kwargs: any):
        if normal_init:
            self.nn_init_ = self.normal_init_
        else:
            self.nn_init_ = nn.init.kaiming_uniform_
        super().__init__(*args, **kwargs)
    
    def init_weights_(self):
        self.layer = nn.Parameter(torch.zeros(
            (self.num_heads, self.head_dim, self.feature_dim),
            dtype=self.dtype, device=self.device,
        ))
        self.nn_init_(self.layer)

    def normal_init_(self, layer: torch.Tensor):
        with torch.no_grad():
            for i in range(layer.shape[0]):
                nn.init.normal_(layer[i])

    def zero_init_with_skip_(self):
        with torch.no_grad():
            nn.init.zeros_(self.layer)

    def zero_init_(self):
        with torch.no_grad():
            for i in range(self.layer.shape[0]):
                nn.init.eye_(self.layer[i])

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Assume x.shape is b h l d"""
        if self.skip_connection:
            return x + torch.einsum('hdf,bhld->bhlf', self.layer, x)  
        return torch.einsum('hdf,bhld->bhlf', self.layer, x)


In [ ]:
class FeatureMap(nn.Module):
    """
    Parent feature map; default is identity function
    """
    def __init__(self, 
                 input_dim: int,                 
                 temp: int = None,
                 head_dim_idx: int = -1, 
                 eps: float = 1e-12, 
                 mlp: nn.Module = None,
                 **kwargs: any):
        super().__init__()
        self.input_dim = input_dim
        self.head_dim_idx = head_dim_idx     
        self.temp = 1. if temp is None else temp
        self.eps = eps
        self.mlp = mlp if mlp is not None else nn.Identity()
        
    def forward(self, x: torch.Tensor):
        """
        Assume x.shape is (batch_size, n_heads, seq_len, head_dim)
        """
        return self.mlp(x)

class FullSpaceMap(nn.Module):
    """
    Project positive features to upper and lower "halfspaces"
    """
    def __init__(self, 
                 head_dim_idx: int = -1, 
                 eps: float = 1e-12,
                 **kwargs: any):
        super().__init__()
        self.head_dim_idx = head_dim_idx
        self.eps = eps
        
    def forward(self, x: torch.Tensor, fmap = None):
        return torch.cat([x, -x], dim=self.head_dim_idx).clamp(min=self.eps)

class ExpDim(FeatureMap):
    """
    Feature maps based on applying exp() element- or dimension-wise
    """
    def __init__(self, 
                 fullspace: bool = True,
                 **kwargs):
        super().__init__(**kwargs)
        self.fs_map = FullSpaceMap(**kwargs)

    def forward(self, x: torch.Tensor):
        x = self.mlp(x)
        return torch.exp(self.fs_map(x * self.temp))

class SoftmaxDim(ExpDim):
    """
    Compute softmax across fullspace
    """
    def __init__(self, *args: any, **kwargs: any):
        super().__init__(*args, **kwargs)
        self.fs_map = None

    def forward(self, x: torch.Tensor):
        x = self.mlp(x)
        x = x * self.temp
        return torch.cat([
            torch.softmax( x, dim=self.head_dim_idx),
            torch.softmax(-x, dim=self.head_dim_idx)
        ], dim=self.head_dim_idx).clamp(min=self.eps)


In [ ]:
class Hedgehog(nn.Module):
    def __init__(self):
        super().__init__()

        layer_kwargs = {
            'num_heads': self.num_heads,
            'head_dim': self.head_dim,
            'dtype': self.q_proj.weight.dtype,
            'device': self.q_proj.weight.device,
        }
        kernel_kwargs = {
            'feature_dim': self.feature_dim,
            'skip_connection': self.skip_connection,
            'zero_init': self.zero_init,
            'bias': self.bias,
        }
        feature_map_kwargs = {
            'input_dim': self.input_dim,
            'eps': 1e-12,
            'fullspace': True,
        }

        UntiedHeadEinsumMLP(**layer_kwargs, **kernel_kwargs)

        learned_kernel = self.init_learned_kernel(learned_kernel, kernel_kwargs)
        self.feature_map_q = SoftmaxDim(mlp=learned_kernel, **feature_map_kwargs)
        self.feature_map_k = copy.deepcopy(self.feature_map_q)
    

    def forward(self, hidden_states: torch.Tensor):
        b, l, _ = hidden_states.size()
        q, k = self.q_proj(hidden_states), self.k_proj(hidden_states)
        q = q.view(b, l, *self.q_shape).transpose(1, 2)
        k = k.view(b, l, *self.k_shape).transpose(1, 2)

        # apply feature map
        q, k = self.feature_map_q(q), self.feature_map_k(k)  
        v = self.v_proj(hidden_states).view(b, l, *self.v_shape).transpose(1, 2)  # b, h, l, d

        # compute linear attention
        y = causal_dot_product(q.contiguous().to(dtype=torch.float32), 
                               k.contiguous().to(dtype=torch.float32), 
                               v.contiguous().to(dtype=torch.float32)).to(dtype=q.dtype)
        y_true = y / (torch.einsum("bhld,bhld->bhl", q, k.cumsum(dim=2))  #  + self.eps
                )[..., None]
    
        # Concatenate heads and apply output projection
        y_true = y_true.transpose(1, 2).contiguous().view(b, l, self.hidden_size)
        y_true = self.o_proj(y_true)
        return y_true
